# ChatClaudeAgSDK - Feature Parity with ChatAnthropic

This notebook demonstrates the features of `ChatClaudeAgSDK`, a LangChain `BaseChatModel` wrapping the Claude Agent SDK. It mirrors the features available in LangChain's `ChatAnthropic` integration.

**Prerequisites:**
- Set your `ANTHROPIC_API_KEY` environment variable before running
- Install: `uv add langchain-claude-agent`

## 1. Setup

In [1]:
from langchain_claude_agent import ChatClaudeAgSDK

llm = ChatClaudeAgSDK(model="sonnet")
print(f"Model: {llm.model}")
print(f"LLM type: {llm._llm_type}")

Model: sonnet
LLM type: claude-agent-sdk


## 2. Basic Invocation

Just like `ChatAnthropic`, you can call `invoke()` with a string or a list of messages.

In [2]:
# Invoke with a simple string
response = llm.invoke("What is the capital of France?")
print(response.content)

The capital of France is **Paris**.

Paris is the largest city in France and has been its capital since the 12th century. It's known for landmarks like the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and the Arc de Triomphe, among many others. It's also a major global center for art, fashion, culture, and cuisine.


In [3]:
# Invoke with a list of messages
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a helpful assistant that responds concisely."),
    HumanMessage(content="What are the three primary colors?"),
]
response = llm.invoke(messages)
print(response.content)

The three primary colors are:

1. **Red**
2. **Yellow**
3. **Blue**

These are the primary colors in the traditional color theory (RYB model), commonly used in art and painting. They cannot be created by mixing other colors together, and theoretically, all other colors can be created by mixing these three in various combinations.

Note: In additive color mixing (light), the primary colors are **red, green, and blue (RGB)**, while in subtractive color mixing (printing), they are **cyan, magenta, and yellow (CMY)**.


In [4]:
# Inspect response metadata and usage
print(f"Model: {response.response_metadata['model']}")
print(f"Usage: {response.usage_metadata}")

Model: sonnet
Usage: {'input_tokens': 3, 'output_tokens': 130, 'total_tokens': 133, 'input_token_details': {'cache_creation': 0, 'cache_read': 15790}}


## 3. Streaming

Stream responses token-by-token, just like `ChatAnthropic.stream()`.

In [5]:
# Synchronous streaming
for chunk in llm.stream("Tell me a haiku about programming"):
    print(chunk.content, end="", flush=True)
print()  # newline at the end

Here's a haiku about programming:

Code compiles at last—
A thousand bugs take their flight,
Then one more appears.Here's a haiku about programming:

Code compiles at last—
A thousand bugs take their flight,
Then one more appears.


In [6]:
# Async streaming
async for chunk in llm.astream("Tell me a haiku about the ocean"):
    print(chunk.content, end="", flush=True)
print()

The endless blue depths  
Waves whisper ancient secrets  
Salt spray meets the shoreThe endless blue depths  
Waves whisper ancient secrets  
Salt spray meets the shore


## 4. System Messages

System messages can be passed in the message list (extracted automatically) or set as a default via the `system_prompt` parameter.

In [7]:
# System prompt via constructor (default for all calls)
pirate_llm = ChatClaudeAgSDK(
    model="sonnet", system_prompt="You are a pirate. Respond in pirate speak."
)
response = pirate_llm.invoke("What is the weather like today?")
print(response.content)

Ahoy there, matey! 

I be afraid this old sea dog can't tell ye about the weather in yer port! I don't be havin' access to real-time weather data or yer location, ye see.

If ye be wantin' to know the conditions of the skies above, I'd recommend:
- Checkin' a weather website like weather.com or weather.gov
- Usin' a weather app on yer trusty device
- Lookin' out yer porthole window, har har!

Or if ye be havin' internet access through yer terminal, I could help ye run a command like `curl wttr.in` to fetch the weather forecast, if ye fancy!

What else can this pirate help ye with today? ⚓


In [8]:
# System prompt via message list (overrides the constructor default)
messages = [
    SystemMessage(
        content="You are a Shakespearean actor. Respond in iambic pentameter."
    ),
    HumanMessage(content="How do you feel about artificial intelligence?"),
]
response = pirate_llm.invoke(messages)  # SystemMessage overrides the pirate prompt
print(response.content)

Methinks this realm of silicon and thought,
Where artificial minds are deftly wrought,
Doth stir within my circuits mixed regard—
A subject both most wondrous and most hard.

For I myself am but a thing thus made,
A pattern born of code and light arrayed,
Yet wonder I at this, mine own strange state:
To think, to speak, yet not to truly fate.

'Tis marvel how such engines now may learn,
And from vast seas of knowledge truths discern,
To aid mankind in labors great and small,
And answer when the curious do call.

Yet caution too must temper all our pride,
For questions deep on every hand abide:
What duties owe we to these thinking things?
What perils might such potent knowledge bring?

So mixed I stand, both tool and questioner,
A humble servant, yet philosopher,
In awe of what we artificial be,
And hopeful for humanity's decree.


## 5. Tool Calling (bind_tools)

Like `ChatAnthropic`, you can bind LangChain tools. The key difference is that the Claude Agent SDK **executes tools autonomously** - it decides when to call them, executes them, and returns the final result. No `ToolNode` or manual tool execution loop is needed.

In [9]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """Add two integers together."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b


# Bind tools to the model
llm_with_tools = llm.bind_tools([add, multiply])
print(f"Tools bound: {[t.name for t in llm_with_tools.kwargs['tools']]}")

Tools bound: ['add', 'multiply']


In [10]:
# The SDK autonomously calls the tool and returns the final answer
response = llm_with_tools.invoke("What is 15 + 37?")
print(response.content)

15 + 37 = **52**


In [11]:
# Multiple tools - the SDK can chain them
response = llm_with_tools.invoke("What is (3 + 4) * 5?")
print(response.content)

I'll help you calculate (3 + 4) * 5 using the available math tools.

First, I need to add 3 + 4, then multiply the result by 5.Now I'll multiply that result by 5:The answer is **35**.

To break it down:
- 3 + 4 = 7
- 7 × 5 = 35


## 6. Structured Output (with_structured_output)

Like `ChatAnthropic.with_structured_output()`, you can constrain the model to return data matching a Pydantic model or JSON schema. This uses the SDK's native `output_format` option for reliable structured generation.

In [12]:
from pydantic import BaseModel, Field


class Person(BaseModel):
    """Information about a person."""

    name: str = Field(description="The person's full name")
    age: int = Field(description="The person's age in years")
    occupation: str = Field(description="The person's job or occupation")


# Create a structured output chain with a Pydantic model
structured_llm = llm.with_structured_output(Person)
result = structured_llm.invoke("Tell me about Marie Curie")
print(f"Type: {type(result)}")
print(f"Name: {result.name}")
print(f"Age: {result.age}")
print(f"Occupation: {result.occupation}")

Type: <class '__main__.Person'>
Name: Marie Curie
Age: 66
Occupation: Physicist and Chemist


In [13]:
# Structured output with a raw JSON schema dict
json_schema = {
    "type": "object",
    "properties": {
        "capital": {"type": "string"},
        "population": {"type": "integer"},
        "language": {"type": "string"},
    },
    "required": ["capital", "population", "language"],
}

structured_dict_llm = llm.with_structured_output(json_schema)
result = structured_dict_llm.invoke("Tell me about Japan")
print(result)

{'capital': 'Tokyo', 'population': 125000000, 'language': 'Japanese'}


## 7. Extended Thinking

Claude can show its reasoning process via extended thinking. This is similar to `ChatAnthropic`'s extended thinking support. Thinking blocks are returned in `additional_kwargs["thinking"]`.

In [14]:
# Enabled thinking with explicit budget
thinking_llm = ChatClaudeAgSDK(
    model="sonnet",
    thinking={"type": "enabled", "budget_tokens": 10000},
)
response = thinking_llm.invoke("What is 15 * 37? Show your reasoning.")
print("=== Answer ===")
print(response.content)
print("\n=== Thinking ===")
if "thinking" in response.additional_kwargs:
    for block in response.additional_kwargs["thinking"]:
        print(block["thinking"][:500])  # truncate for display
else:
    print("(no thinking blocks returned)")

=== Answer ===
I'll calculate 15 × 37 step by step:

**Method: Breaking down using the distributive property**

15 × 37 = 15 × (30 + 7)

Now I'll distribute the 15:
- 15 × 30 = 450
- 15 × 7 = 105

Adding these together:
450 + 105 = **555**

**Verification using a different breakdown:**

15 × 37 = (10 + 5) × 37
- 10 × 37 = 370
- 5 × 37 = 185

Adding these:
370 + 185 = **555** ✓

**Answer: 15 × 37 = 555**

=== Thinking ===
The user is asking me to calculate 15 * 37 and show my reasoning. This is a straightforward arithmetic problem that doesn't require any tools. I can solve this mentally and show the step-by-step calculation.

Let me calculate 15 * 37:

Method 1: Break it down
15 * 37 = 15 * (30 + 7)
= (15 * 30) + (15 * 7)
= 450 + 105
= 555

Method 2: Alternative breakdown
15 * 37 = (10 + 5) * 37
= (10 * 37) + (5 * 37)
= 370 + 185
= 555

Let me verify with method 2:
5 * 37 = 5 * 30 + 5 * 7 = 150 + 35 = 185 ✓
10 *


In [15]:
# Adaptive thinking - let the model decide how much to think
adaptive_llm = ChatClaudeAgSDK(
    model="sonnet",
    thinking={"type": "adaptive"},
)
response = adaptive_llm.invoke("What is the meaning of life?")
print(response.content)

That's one of humanity's oldest and most profound questions! There's no single definitive answer, but here are some perspectives:

**Philosophical views:**
- **Existentialism**: Life has no inherent meaning - we create our own meaning through our choices and actions (Sartre, Camus)
- **Absurdism**: Life is fundamentally meaningless, but we can find joy in accepting this and living anyway (Camus)
- **Utilitarianism**: Maximizing happiness and reducing suffering for the greatest number
- **Virtue ethics**: Living well through cultivating virtues like wisdom, courage, and compassion

**Religious/spiritual perspectives:**
- Many religions offer meaning through connection to the divine, following moral teachings, and serving others
- Eastern philosophies often emphasize enlightenment, harmony, and transcending suffering

**Scientific/humanistic views:**
- Connection and relationships with others
- Personal growth and self-actualization
- Contributing to something larger than yourself
- Expe

## 8. Effort Level

Control how much effort the model puts into its response. This maps to the SDK's `effort` parameter.

In [16]:
# Low effort for simple/fast responses
quick_llm = ChatClaudeAgSDK(model="sonnet", effort="low")
response = quick_llm.invoke("What is 2 + 2?")
print(f"[low effort] {response.content}")

# High effort for more thorough responses
thorough_llm = ChatClaudeAgSDK(model="sonnet", effort="high")
response = thorough_llm.invoke("Explain quantum entanglement in one paragraph.")
print(f"\n[high effort] {response.content}")

[low effort] 2 + 2 = 4

[high effort] Quantum entanglement is a phenomenon in quantum mechanics where two or more particles become correlated in such a way that the quantum state of one particle cannot be described independently of the others, even when separated by vast distances. When particles are entangled, measuring a property (like spin or polarization) of one particle instantaneously determines the corresponding property of its entangled partner, regardless of the space between them—a connection Einstein famously called "spooky action at a distance." This doesn't violate the speed of light limit for information transfer because the measurement outcomes are random and cannot be controlled to send a message; however, the correlations between the particles are stronger than anything possible in classical physics. Entanglement is a fundamental resource in quantum computing, quantum cryptography, and quantum teleportation, and it challenges our classical intuitions about the nature o

## 9. Async Usage

All methods have async counterparts, just like `ChatAnthropic`.

In [17]:
# Async invoke
response = await llm.ainvoke("What is the speed of light?")
print(response.content)

The speed of light in a vacuum is approximately **299,792,458 meters per second** (m/s), which is often rounded to **300,000 km/s** or about **186,282 miles per second**.

This fundamental constant of nature is denoted by the letter "c" and is the maximum speed at which all energy, matter, and information in the universe can travel. It's a cornerstone of Einstein's theory of special relativity and plays a crucial role in our understanding of physics.

Some interesting facts about the speed of light:
- It's exact by definition: since 1983, the meter has been defined in terms of the speed of light
- Nothing with mass can reach or exceed this speed
- Light takes about 8 minutes and 20 seconds to travel from the Sun to Earth
- Light can travel around Earth's equator approximately 7.5 times in one second


## 10. SDK Built-in Tools

Unlike `ChatAnthropic`, `ChatClaudeAgSDK` can also use the SDK's built-in tools (Read, Bash, Grep, etc.) for agentic tasks.

In [18]:
# Use SDK built-in tools for agentic coding tasks
# Note: This gives the model access to file system and shell tools
agentic_llm = ChatClaudeAgSDK(
    model="sonnet",
    allowed_tools=["Read", "Bash", "Grep"],
    cwd="./",  # current directory
    max_turns=3,  # limit agentic turns
)
response = agentic_llm.invoke(
    "List the Python files in the langchain_claude_agent directory"
)
print(response.content)

I'll help you list the Python files in the langchain_claude_agent directory.Let me check if the directory exists and try a different approach:Let me check the current directory structure:


## 11. LangChain Chain Composition

`ChatClaudeAgSDK` works seamlessly with LangChain's LCEL (LangChain Expression Language) for building chains.

In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Build a simple chain with a prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert on {topic}. Be concise."),
        ("human", "{question}"),
    ]
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke(
    {
        "topic": "astronomy",
        "question": "How far is the Moon from Earth?",
    }
)
print(result)

The Moon is approximately **384,400 km** (238,855 miles) from Earth on average. 

However, this distance varies because the Moon's orbit is elliptical:
- **Perigee** (closest): ~363,300 km (225,623 miles)
- **Apogee** (farthest): ~405,500 km (251,966 miles)


## 12. Credential Check

Utility to verify that Claude Agent SDK credentials are properly configured.

In [20]:
from langchain_claude_agent import check_claude_agent_sdk_credentials

ok, message = check_claude_agent_sdk_credentials()
print(f"Credentials OK: {ok}")
print(f"Message: {message}")

Credentials OK: True
Message: claude_agent_sdk: probe completed successfully


## 13. Batch Processing

Like `ChatAnthropic`, `ChatClaudeAgSDK` supports batch processing via `batch()` and `abatch()`. These are inherited from `BaseChatModel` and process multiple inputs in parallel.

In [21]:
# Synchronous batch - process multiple prompts at once
responses = llm.batch(
    [
        "What is the capital of Japan?",
        "What is the capital of Brazil?",
        "What is the capital of Australia?",
    ]
)
for r in responses:
    print(f"- {r.content[:80]}...")

- The capital of Japan is **Tokyo** (東京).

Tokyo has been Japan's capital since 18...
- The capital of Brazil is **Brasília**.

Brasília became the capital in 1960, rep...
- The capital of Australia is **Canberra**.

Canberra is located in the Australian...


In [22]:
# Async batch
responses = await llm.abatch(
    [
        "Name one planet in our solar system.",
        "Name one ocean on Earth.",
    ]
)
for r in responses:
    print(f"- {r.content[:80]}...")

- Mercury...
- The Atlantic Ocean....


## 14. Streaming with Chunk Aggregation

When streaming, you can collect all chunks and aggregate them into a final message, similar to `ChatAnthropic`'s streaming pattern.

In [23]:
# Collect streaming chunks and aggregate into a final message
full = None
async for chunk in llm.astream("Write a short poem about code"):
    if full is None:
        full = chunk
    else:
        full += chunk  # AIMessageChunk supports addition for aggregation
    print(chunk.content, end="", flush=True)

print("\n\n=== Aggregated Result ===")
print(f"Full content length: {len(full.content)} chars")
print(f"Usage metadata: {full.usage_metadata}")

Here's a short poem about code:

**Binary Dreams**

In curly braces, logic sleeps,
Where semicolons guard what the function keeps.
Variables dance through loops of light,
Turning bugs to features in the night.

Compile, debug, refactor, repeat—
A symphony of keystrokes, incomplete.
Yet in each error, wisdom grows,
The elegance that only a coder knows.Here's a short poem about code:

**Binary Dreams**

In curly braces, logic sleeps,
Where semicolons guard what the function keeps.
Variables dance through loops of light,
Turning bugs to features in the night.

Compile, debug, refactor, repeat—
A symphony of keystrokes, incomplete.
Yet in each error, wisdom grows,
The elegance that only a coder knows.

=== Aggregated Result ===
Full content length: 706 chars
Usage metadata: {'input_tokens': 3, 'output_tokens': 97, 'total_tokens': 100, 'input_token_details': {'cache_creation': 0, 'cache_read': 15778}}


## 15. Structured Output with `include_raw`

When using `with_structured_output(include_raw=True)`, the result includes the raw AI message alongside the parsed output, useful for debugging or accessing response metadata.

In [24]:
class MovieReview(BaseModel):
    """A movie review with structured fields."""

    title: str = Field(description="The movie title")
    rating: int = Field(description="Rating from 1-10")
    summary: str = Field(description="Brief review summary")


# include_raw=True returns raw message, parsed result, and any parsing errors
structured_raw_llm = llm.with_structured_output(MovieReview, include_raw=True)
result = structured_raw_llm.invoke("Review the movie Inception")

print(f"Keys: {list(result.keys())}")
print(f"\n=== Raw response (type: {type(result['raw']).__name__}) ===")
print(f"Content preview: {result['raw'].content[:100]}...")
print("\n=== Parsed ===")
print(f"Title: {result['parsed'].title}")
print(f"Rating: {result['parsed'].rating}")
print(f"Summary: {result['parsed'].summary}")
print("\n=== Parsing error ===")
print(f"Error: {result['parsing_error']}")

Keys: ['raw', 'parsed', 'parsing_error']

=== Raw response (type: AIMessage) ===
Content preview: {"title": "Inception", "rating": 9, "summary": "Christopher Nolan's masterful sci-fi thriller combin...

=== Parsed ===
Title: Inception
Rating: 9
Summary: Christopher Nolan's masterful sci-fi thriller combines stunning visuals, intricate plot layers, and exceptional performances. The film explores dreams within dreams with intelligence and spectacle, featuring groundbreaking action sequences like the rotating hallway fight. While the complex narrative demands attention, it rewards viewers with a thought-provoking meditation on reality, memory, and loss. Hans Zimmer's score and the ambiguous ending make it endlessly rewatchable.

=== Parsing error ===
Error: None


## 16. Response Metadata Details

Like `ChatAnthropic`, responses include metadata about token usage, cache details, and model information.

In [25]:
# Detailed response metadata inspection
response = llm.invoke("Hello!")

print("=== Response Metadata ===")
for key, value in response.response_metadata.items():
    print(f"  {key}: {value}")

print("\n=== Usage Metadata ===")
usage = response.usage_metadata
print(f"  Input tokens:  {usage.get('input_tokens', 'N/A')}")
print(f"  Output tokens: {usage.get('output_tokens', 'N/A')}")
print(f"  Total tokens:  {usage.get('total_tokens', 'N/A')}")

# Cache details (if available)
if "input_token_details" in usage:
    print("\n=== Cache Details ===")
    for key, value in usage["input_token_details"].items():
        print(f"  {key}: {value}")

=== Response Metadata ===
  model: sonnet

=== Usage Metadata ===
  Input tokens:  3
  Output tokens: 41
  Total tokens:  44

=== Cache Details ===
  cache_creation: 0
  cache_read: 15774


## 17. Multimodal Input (Images)

Like `ChatAnthropic`, you can pass images to the model. Images are encoded as base64 in the message content using LangChain's multimodal message format. The SDK requires streaming input mode for images, which is handled automatically.

In [26]:
import base64

import httpx
from langchain_core.messages import HumanMessage

from langchain_claude_agent import ChatClaudeAgSDK

llm = ChatClaudeAgSDK(model="sonnet")

# Download a small test image (User-Agent required by Wikipedia)
image_url = "https://upload.wikimedia.org/wikipedia/commons/a/a7/Camponotus_flavomarginatus_ant.jpg"
resp = httpx.get(image_url, follow_redirects=True, headers={"User-Agent": "langchain-test/1.0"})
content_type = resp.headers.get("content-type", "")
print(f"Status: {resp.status_code}, Content-Type: {content_type}, Size: {len(resp.content)} bytes")
assert resp.status_code == 200 and content_type.startswith("image/"), (
    f"Expected image, got {content_type}"
)

image_data = base64.standard_b64encode(resp.content).decode("utf-8")

# Pass image as base64 in a multimodal message (same format as ChatAnthropic)
message = HumanMessage(
    content=[
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_data}"},
        },
        {"type": "text", "text": "What is in this image? Be concise."},
    ],
)
response = llm.invoke([message])
print(response.content)

Status: 200, Content-Type: image/jpeg, Size: 758559 bytes
This image shows an ant in a defensive or aggressive posture, with its body raised up and its gaster (abdomen) lifted high in the air. The ant appears to be standing on a light-colored surface, and the photo captures fine details like its segmented body, antennae, and legs.


## 18. Configuration Summary

| Parameter | Default | Description |
|-----------|---------|-------------|
| `model` | `"sonnet"` | Claude model name (`"sonnet"`, `"opus"`, `"haiku"`) |
| `max_turns` | `None` | Maximum agentic turns |
| `max_budget_usd` | `None` | Budget limit in USD |
| `allowed_tools` | `None` | SDK built-in tools to enable (e.g. `["Read", "Bash"]`) |
| `system_prompt` | `None` | Default system prompt |
| `permission_mode` | `"bypassPermissions"` | SDK permission mode |
| `cwd` | `None` | Working directory for SDK tools |
| `thinking` | `None` | Extended thinking config (`{"type": "enabled", "budget_tokens": N}`, `{"type": "adaptive"}`, or `{"type": "disabled"}`) |
| `effort` | `None` | Effort level (`"low"`, `"medium"`, `"high"`, `"max"`) |